# Module 07.2 — HealthScore_IO

Este notebook ejecuta el módulo `07_2_healthscore_io.py` usando directamente:

- el `08_Perf_*.csv` generado por el Módulo 06;
- el `4_Execution_Summary_*.csv` generado por el Módulo 05.

La implementación sigue la formulación final de la tesis:

\[
HealthScore_{IO}=
rac{10}{5}(BW_{norm}+IOPS_{norm}+T_{io,gain}+T_{run,gain}+\Gamma_{gain})
\]

La normalización se realiza de forma independiente para cada grupo experimental y `Access_Mode` no se utiliza para inferir el tipo de escalamiento.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from importlib.util import spec_from_file_location, module_from_spec

MODULE_DIR = Path("./Module").resolve()
MODULE07_2_FILE = MODULE_DIR / "07_2_healthscore_io.py"

if not MODULE07_2_FILE.exists():
    raise FileNotFoundError(f"No se encontró Module 06: {MODULE07_2_FILE}")

spec = spec_from_file_location("deeptuneio_module07", MODULE07_2_FILE)
module07 = module_from_spec(spec)
sys.modules[spec.name] = module07
spec.loader.exec_module(module07)

print("Module 07:", MODULE07_2_FILE)
print("Version  :", module07.VERSION)


Module 07: F:\NoteBooks\CORA_DR_UAB\Tesis\Scripts\DeepTuneIO\07_Analysis\07_2_HealthScore_IO\Module\07_2_healthscore_io.py
Version  : 2.2.0


In [2]:
# ==============================================================================
# 1. Selección de campaña
# ============================================================================== 
CAMPAIGNS = {
    "DeepGalaxy": {
        "DG_bw512_f3c5x64": {
            "scenarios": [
                "lustre_1ost",
                "lustre_2ost",
                "lustre_4ost",
                "nfs_SnGPU_e1",
            ],
        },
    },
    "DLIOv1": {
        "HDF5_64bs": {"scenarios": ["lustre_ost"]},
        "NPZ_64bs": {"scenarios": ["lustre_ost"]},
        "TFRecord_256kts_64bs": {"scenarios": ["lustre_ost"]},
        "TFRecord_1mts_64bs": {"scenarios": ["lustre_ost"]},
    },
}

APPLICATION_POS = 1
DATASET_POS = 1
SCENARIO_POS = 4

APPLICATION = list(CAMPAIGNS)[APPLICATION_POS - 1]
DATASET_GROUP = list(CAMPAIGNS[APPLICATION])[DATASET_POS - 1]
SCENARIO = CAMPAIGNS[APPLICATION][DATASET_GROUP]["scenarios"][SCENARIO_POS - 1]

print("Application   :", APPLICATION)
print("Dataset group :", DATASET_GROUP)
print("Scenario      :", SCENARIO)


Application   : DeepGalaxy
Dataset group : DG_bw512_f3c5x64
Scenario      : nfs_SnGPU_e1


In [3]:
# ==============================================================================
# 2. Raíz de la campaña y rutas
# ============================================================================== 

PROJECT_ROOT = Path(r"D:\DL_Experiments\CORA_DR_UAB").resolve()
ARTICLE_ID = "Tesis"

ARTICLE_ROOT = PROJECT_ROOT / ARTICLE_ID
CAMPAIGN_ROOT = ARTICLE_ROOT / "Data" / APPLICATION / DATASET_GROUP / SCENARIO

print("Project root  :", PROJECT_ROOT)
print("Article root  :", ARTICLE_ROOT)
print("Campaign root :", CAMPAIGN_ROOT)
print("Exists        :", CAMPAIGN_ROOT.exists())

Project root  : D:\DL_Experiments\CORA_DR_UAB
Article root  : D:\DL_Experiments\CORA_DR_UAB\Tesis
Campaign root : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1
Exists        : True


In [4]:

EVENT = "Tesis"

RESULTS_DIR = CAMPAIGN_ROOT / "results" 
PERF_CSV = RESULTS_DIR / "Perf" / f"08_Perf_{APPLICATION}_{SCENARIO}_{DATASET_GROUP}_{EVENT}.csv"
PARSER_CSV = RESULTS_DIR / "Parser" / f"4_Execution_Summary_{APPLICATION}_Parser_{SCENARIO}_{DATASET_GROUP}_{EVENT}.csv"

OUTPUT_DIR = RESULTS_DIR / "Analysis" / "healthscore"

print("Campaign root :", CAMPAIGN_ROOT)
print("PERF          :", PERF_CSV)
print("PARSER        :", PARSER_CSV)
print("Output dir    :", OUTPUT_DIR)


Campaign root : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1
PERF          : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Perf\08_Perf_DeepGalaxy_nfs_SnGPU_e1_DG_bw512_f3c5x64_Tesis.csv
PARSER        : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Parser\4_Execution_Summary_DeepGalaxy_Parser_nfs_SnGPU_e1_DG_bw512_f3c5x64_Tesis.csv
Output dir    : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Analysis\healthscore


In [5]:
# ==============================================================================
# 3. Ejecutar HealthScore_IO
# ==============================================================================

return_code = module07.main([
    "--perf-csv", str(PERF_CSV),
    "--parser-csv", str(PARSER_CSV),
    "--output-dir", str(OUTPUT_DIR),
    "--application", APPLICATION,
    "--dataset-group", DATASET_GROUP,
    "--scenario", SCENARIO,
    "--event", EVENT,
])

print("Return code:", return_code)


DeepTuneIO Analyzer — Module 07.2 HealthScore_IO v2.2.0
Application          : DeepGalaxy
Dataset group        : DG_bw512_f3c5x64
Scenario             : nfs_SnGPU_e1
Event                : Tesis
Normalization group  : ['Application', 'Dataset_Group', 'Scenario', 'File_Format', 'Access_Mode', 'Scale_Type']
Aggregated configs   : 12
Excluded processes  : none

Outputs:
  D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Analysis\healthscore\07_2_healthscore_io_DeepGalaxy_DG_bw512_f3c5x64_nfs_SnGPU_e1_Tesis.csv
  D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Analysis\healthscore\07_2_healthscore_io_ranked_DeepGalaxy_DG_bw512_f3c5x64_nfs_SnGPU_e1_Tesis.csv
  D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Analysis\healthscore\07_2_healthscore_io_summary_DeepGalaxy_DG_bw512_f3c5x64_nfs_SnGPU_e1_Tesis.csv
  D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_

In [6]:
# ==============================================================================
# 4. Revisar resultados
# ==============================================================================

stem = f"{APPLICATION}_{DATASET_GROUP}_{SCENARIO}_{EVENT}"
ranked_csv = OUTPUT_DIR / f"07_2_healthscore_io_ranked_{stem}.csv"

health_df = pd.read_csv(ranked_csv)

display_cols = [
    "HealthScore_Rank",
    "Access_Mode",
    "Scale_Type",
    "Nodes",
    "Processes_IO",
    "n_runs",
    "Bandwidth_MiB_s",
    "IOPS",
    "IO_Time_s",
    "Run_Time_s",
    "Gamma_IO_pct",
    "BW_norm",
    "IOPS_norm",
    "Tio_gain",
    "Trun_gain",
    "Gamma_gain",
    "HealthScore_IO",
]

display(health_df[[c for c in display_cols if c in health_df.columns]])


,HealthScore_Rank,Access_Mode,Scale_Type,Processes_IO,n_runs,Bandwidth_MiB_s,IOPS,IO_Time_s,Run_Time_s,Gamma_IO_pct,BW_norm,IOPS_norm,Tio_gain,Trun_gain,Gamma_gain,HealthScore_IO
0,1,0,ss,64,2,437.524374,1.282255e+05,5.491261,2413.078700,0.227562,1.000000,1.000000,0.986952,1.000000,0.000000,7.973904
1,2,0,ss,48,2,402.983796,1.149564e+05,5.461752,3347.132300,0.163177,0.921055,0.896517,0.992284,0.720939,0.282935,7.627460
2,3,0,ss,32,3,399.687870,1.099081e+05,5.419610,4434.802367,0.122206,0.913521,0.857147,1.000000,0.544123,0.462977,7.555537
3,4,0,ss,16,2,376.077687,9.967469e+04,5.470731,8151.851750,0.067110,0.859558,0.777339,0.990656,0.296016,0.705091,7.257319
4,5,0,ss,8,3,227.447640,5.902912e+04,8.907010,14416.008300,0.061786,0.519851,0.460354,0.608466,0.167389,0.728490,4.969099
5,6,0,ss,4,2,220.122543,5.651053e+04,9.081259,27147.211750,0.033452,0.503109,0.440712,0.596791,0.088889,0.852999,4.964999
6,1,1,ss,64,1,16538.709796,4.209012e+06,9.722719,2027.502800,0.479542,1.000000,1.000000,1.000000,1.000000,0.305938,8.611875
7,2,1,ss,32,1,7910.351223,2.012282e+06,13.937074,3802.774200,0.366498,0.478293,0.478089,0.697616,0.533164,0.469551,5.313426
8,3,1,ss,16,3,3711.247712,9.436454e+05,16.097930,7896.221967,0.203869,0.224398,0.224196,0.603973,0.256769,0.704931,4.028535
9,4,1,ss,48,1,6352.899860,1.616329e+06,22.355524,3235.617200,0.690920,0.384123,0.384016,0.434913,0.626620,0.000000,3.659346


In [7]:
# ==============================================================================
# 5. Mejor configuración por grupo experimental
# ==============================================================================

best = health_df[health_df["HealthScore_Rank"] == 1].copy()
display(best[[
    c for c in [
        "Application", "Dataset_Group", "Scenario", "File_Format",
        "Access_Mode", "Scale_Type", "Processes_IO", "HealthScore_IO"
    ] if c in best.columns
]])


,Application,Dataset_Group,Scenario,File_Format,Access_Mode,Scale_Type,Processes_IO,HealthScore_IO
0,DeepGalaxy,DG_bw512_f3c5x64,nfs_SnGPU_e1,hdf5,0,ss,64,7.973904
6,DeepGalaxy,DG_bw512_f3c5x64,nfs_SnGPU_e1,hdf5,1,ss,64,8.611875
